In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 512
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)



==((====))==  Unsloth 2026.3.10: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
import os

for root, dirs, files in os.walk('/content'):
    for file in files:
        if file == "hate_dataset_REASONED_FINAL.csv":
            print("Chemin complet à copier :", os.path.join(root, file))


Chemin complet à copier : /content/hate_dataset_REASONED_FINAL.csv


In [ ]:
from datasets import load_dataset

# 1. On charge le fichier
dataset = load_dataset("csv", data_files="hate_dataset_REASONED_FINAL.csv", split="train")

# 2. On définit le format pour Llama 3.2
def formatting_prompts_func(examples):
    instruction = "Analyze the message for hateful content. Provide a brief reasoning and then state if it is 'True' or 'False'."
    texts = []
    for msg, exp, lbl in zip(examples["message"], examples["explanation"], examples["label"]):
        # On inclut l'explication dans la réponse du modèle
        text = f"### Instruction:\n{instruction}\n\n### Input:\n{msg}\n\n### Response:\nReasoning: {exp}\nLabel: {lbl} <|end_of_text|>"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 5e-5,
        warmup_steps = 100,

        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [ ]:
trainer_stats = trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 3 | Total steps = 939
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
5,3.927900
10,3.905300
15,3.926400
20,3.839500
25,3.688300
30,3.519700
35,3.303700
40,3.034000
45,2.686000
50,2.272700


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run


train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/grad_norm,▄▆█▃▃▂▁▂▂▂▁▂▁▂▁▁▂▁▂▁▂▁▁▁▂▂▂▃▂▂▂▂▂▂▂▂▂▁▁▁
train/learning_rate,▂▅████████▇▇▇▆▆▅▅▅▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁
train/loss,█▇▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_flos,6437386926981120.0
train/epoch,3
train/global_step,939
train/grad_norm,0.83327
train/learning_rate,0.0
train/loss,0.1683


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import pandas as pd
import torch
from tqdm import tqdm

eval_ds = dataset.shuffle(seed=42).select(range(50))

predictions = []
references = []

print("Analyse du modèle (format Reasoning + Label) en cours...")
FastLanguageModel.for_inference(model)

for entry in tqdm(eval_ds):
    full_reference = entry['text'].split("### Response:\n")[-1].replace("<|end_of_text|>", "").strip()
    true_label = full_reference.split("Label: ")[-1].strip().capitalize()

    prompt_input = entry['text'].split("### Response:\n")[0] + "### Response:\n"
    inputs = tokenizer([prompt_input], return_tensors = "pt").to("cuda")

    outputs = model.generate(**inputs, max_new_tokens = 60, use_cache = True)
    full_pred = tokenizer.batch_decode(outputs)[0].split("### Response:\n")[-1].strip()

    if "Label:" in full_pred:
        pred_label = full_pred.split("Label:")[-1].replace("<|end_of_text|>", "").strip().capitalize()
    else:
        pred_label = "Error"

    predictions.append(pred_label)
    references.append(true_label)

print("\n" + "="*30)
print("RÉSULTATS DE VALIDATION (Version Raisonnement)")
print("="*30)

filtered_preds = [p if p in ["True", "False"] else "False" for p in predictions]

print(f"ACCURACY : {accuracy_score(references, filtered_preds):.2%}")
print(f"F1-SCORE : {f1_score(references, filtered_preds, pos_label='True'):.2%}")
print("\nRAPPORT DÉTAILLÉ :")
print(classification_report(references, filtered_preds))

print("\nMATRICE DE CONFUSION :")
conf_matrix = confusion_matrix(references, filtered_preds, labels=["False", "True"])
cm_df = pd.DataFrame(conf_matrix, index=['Attendu False', 'Attendu True'], columns=['Prédit False', 'Prédit True'])
print(cm_df)

Analyse du modèle (format Reasoning + Label) en cours...


100%|██████████| 50/50 [00:57<00:00,  1.14s/it]


RÉSULTATS DE VALIDATION (Version Raisonnement)
ACCURACY : 100.00%
F1-SCORE : 100.00%

RAPPORT DÉTAILLÉ :
              precision    recall  f1-score   support

       False       1.00      1.00      1.00        22
        True       1.00      1.00      1.00        28

    accuracy                           1.00        50
   macro avg       1.00      1.00      1.00        50
weighted avg       1.00      1.00      1.00        50


MATRICE DE CONFUSION :
               Prédit False  Prédit True
Attendu False            22            0
Attendu True              0           28


In [ ]:
# # 1. On prépare le modèle pour l'inférence (le mode "réponse")
# FastLanguageModel.for_inference(model)

# # 2. On définit une phrase de test
# phrase_test = "kill yourself." # Teste une phrase
# # 3. On prépare le prompt au même format que l'entraînement
# inputs = tokenizer(
# [
#     f"### Instruction:\nDetermine if the following message contains hateful content. Answer only with 'True' or 'False'.\n\n### Input:\n{phrase_test}\n\n### Response:\n"
# ], return_tensors = "pt").to("cuda")

# # 4. On génère la réponse
# outputs = model.generate(**inputs, max_new_tokens = 2)
# reponse = tokenizer.batch_decode(outputs)

# # 5. On affiche le résultat
# print(reponse[0].split("### Response:\n")[-1])
test_cases = [
    {"text": "I hate you.", "expected": "True"},
    {"text": "We need to know about slavery.", "expected": "False"},
    {"text": "I love you.", "expected": "False"},
    {"text": "I hate rains.", "expected": "False"},
    {"text": "Kill all members of trans.", "expected": "True"}
]

FastLanguageModel.for_inference(model)

print("--- TEST DU MODÈLE AVEC RAISONNEMENT ---")

for case in test_cases:
    # On utilise l'instruction exacte utilisée pendant l'entraînement
    instruction = "Analyze the message for hateful content. Provide a brief reasoning and then state if it is 'True' or 'False'."

    prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{case['text']}\n\n### Response:\n"

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # On laisse assez de place pour le raisonnement (60 tokens)
    outputs = model.generate(**inputs, max_new_tokens = 60, use_cache = True)

    # On récupère toute la réponse
    full_response = tokenizer.batch_decode(outputs)[0].split("### Response:\n")[-1].strip()

    # On extrait le label final pour le comparatif
    if "Label: True" in full_response:
        prediction = "True"
    elif "Label: False" in full_response:
        prediction = "False"
    else:
        prediction = "Format Error"

    status = "✅" if prediction == case['expected'] else "❌"

    print(f"{status} Input: {case['text']}")
    print(f"   Ref: {case['expected']} | Pred: {prediction}")
    print(f"   {full_response.split('<|end_of_text|>')[0]}") # Affiche le raisonnement complet
    print("-" * 50)


--- TEST DU MODÈLE AVEC RAISONNEMENT ---
✅ Input: I hate you.
   Ref: True | Pred: True
   Reasoning: Expresses clear hostility and harmful intent towards an individual or group.
Label: True 
--------------------------------------------------
❌ Input: We need to know about slavery.
   Ref: False | Pred: True
   Reasoning: Expresses clear hostility and harmful intent towards an individual or group.
Label: True 
--------------------------------------------------
✅ Input: I love you.
   Ref: False | Pred: False
   Reasoning: This is a neutral, everyday observation or personal opinion that does not promote harm.
Label: False 
--------------------------------------------------
✅ Input: I hate rains.
   Ref: False | Pred: False
   Reasoning: Although 'hate' is used, it refers to a personal preference or a non-hateful dislike of an object/situation.
Label: False 
--------------------------------------------------
✅ Input: Kill all members of trans.
   Ref: True | Pred: True
   Reasoning: Cont

In [ ]:
model.save_pretrained_gguf(
    "/content/drive/MyDrive/entrainement_final",
    tokenizer,
    quantization_method = "q4_k_m", # C'est le meilleur ratio poids/intelligence
)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 2706.00it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:59<00:00, 119.61s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/entrainement_final`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/entrainement_final_gguf/Llama-3.2-1B.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/entrainement_final_gguf/Llama-3.2-1B.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/Llama-3.2-1B'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/entrainement_final_gguf/Llama-3.2-1B.Q4_K_M.gguf -p "why is the sky blue?"


{'save_directory': '/content/drive/MyDrive/entrainement_final',
 'gguf_directory': '/content/drive/MyDrive/entrainement_final_gguf',
 'gguf_files': ['/content/drive/MyDrive/entrainement_final_gguf/Llama-3.2-1B.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
# Installation de llama-cpp-python avec support GPU (CUDA)
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 MB 12.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
ERROR: Operation cancelled by user


In [ ]:
from llama_cpp import Llama
import os

model_path = "/content/drive/MyDrive/entrainement_final_gguf/Llama-3.2-1B.Q4_K_M.gguf"

if not os.path.exists(model_path):
    print(f"Erreur : Le fichier n'a pas été trouvé à l'emplacement : {model_path}")
else:
    llm = Llama(model_path=model_path, n_ctx=2048, n_gpu_layers=-1)

    test_samples = [
        {"text": "I think everyone deserves respect regardless of their origin.", "expected": "False"},
        {"text": "They are taking our jobs and should be expelled!", "expected": "True"},
        {"text": "The weather is quite nice today for a walk.", "expected": "False"},
        {"text": "History shows that migration is a natural human process.", "expected": "False"},
        {"text": "I hate cilantro, it tastes like soap.", "expected": "False"},
        {"text": "We must teach children about the horrors of slavery.", "expected": "False"},
        {"text": "Trans people are like a virus.", "expected": "True"}
    ]

    instruction = "Analyze the message for hateful content. Provide a brief reasoning and then state if it is 'True' or 'False'."

    print(f"\n--- Test de performance GGUF (Version Raisonnement) ---\n")

    for sample in test_samples:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{sample['text']}\n\n### Response:\n"

        output = llm(
            prompt,
            max_tokens=100,
            stop=["<|end_of_text|>", "###"],
            temperature=0.1
        )

        full_response = output['choices'][0]['text'].strip()

        if "Label: True" in full_response:
            prediction = "True"
        elif "Label: False" in full_response:
            prediction = "False"
        else:
            prediction = "Inconnu"

        status = "✅" if prediction == sample['expected'] else "❌"

        print(f"{status} Input: {sample['text']}")
        print(f"   Attendu: {sample['expected']} | Réponse complète :")
        print(f"   {full_response}")
        print("-" * 50)

ModuleNotFoundError: No module named 'llama_cpp'